# Week 1 Day 5 Lab 1 - A Sonnet manager with Haiku workers

My change for the Lab 1 assignment: the main agent stays on Claude Sonnet 5.5, and the claim-handler subagents run on Claude Haiku 5.5, the smallest and cheapest Claude model.

Why this change: the subagents do nearly all the reading. Each one reads the policy wording, the claims manual, the claim, the policy and the customer. The main agent only hands out the claims, keeps the task list and writes the handover. So the workers should spend most of the tokens, and that's where a cheaper model should save the most.

**The question: does a cheaper model for the workers cut the cost without losing accuracy?**

We work the same five claims twice, once with every agent on Sonnet as in the lab, and once with Haiku workers, then compare accuracy, cost and time.

## 4 parts:

PART 1: The claims desk

PART 2: The one change

PART 3: Working the queue twice

PART 4: How they compare

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">Like Lab 1, this uses Anthropic's API, with <code>ANTHROPIC_API_KEY</code> in your <code>.env</code> file. It works the queue twice, so both runs together cost about 50 cents.
            </span>
        </td>
    </tr>
</table>

In [ ]:
import json
import os
import shutil
import time
from pathlib import Path

import pandas as pd
from claude_agent_sdk import AgentDefinition, AssistantMessage, ClaudeAgentOptions, ResultMessage, TextBlock, ToolUseBlock, query
from dotenv import load_dotenv
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)

if os.getenv("ANTHROPIC_API_KEY"):
    print("Anthropic API key found")
else:
    print("No Anthropic API key found. Add ANTHROPIC_API_KEY to the .env file")

## Part 1: The claims desk

The same five claims as the lab. This notebook sits two folders down in `community_contributions/erfan_hesami`, so the data is at `../../data`.

Each run gets a desk of its own, so the two runs' decisions never mix. The desks are in this folder's `.gitignore`.

In [ ]:
DATA = Path("../../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels


claims, labels = get_sample_data(tiny=True)

for claim in claims:
    print(claim["claim_id"], claim["summary"])

In [ ]:
def make_desk(desk, claims):
    """Build a fresh claims desk with the claims to decide and everything needed to decide them."""
    shutil.rmtree(desk, ignore_errors=True)
    for folder in ["claims", "policies", "customers", "decisions"]:
        (desk / folder).mkdir(parents=True)
    for claim in claims:
        (desk / "claims" / f"{claim['claim_id']}.json").write_text(json.dumps(claim, indent=2), encoding="utf-8")
    for policy in load("policies.json"):
        (desk / "policies" / f"{policy['policy_id']}.json").write_text(json.dumps(policy, indent=2), encoding="utf-8")
    for customer in load("customers.json"):
        (desk / "customers" / f"{customer['customer_id']}.json").write_text(json.dumps(customer, indent=2), encoding="utf-8")
    for name in ["policy_wording.md", "claims_manual.md", "company.json"]:
        shutil.copy(DATA / name, desk / name)

The instructions are exactly the lab's, so the model is the only thing that changes.

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance, working through the queue of new claims on your own.

Everything you need is in this folder:
- claims/ has one file for each claim waiting in the queue
- policies/ and customers/ have the policy records and the customer records, one file each, named by their ids
- policy_wording.md is the policy wording and claims_manual.md is the claims handling manual
- company.json has the company's limits

Decide every claim in claims/. Keep a task list with one task for each claim, and give each claim to the claim-handler agent, which decides it and writes its decision file.
When every claim has a decision, write handover.md at the top of this folder: a short note for the handler on what was decided and anything they should know.
"""

HANDLER_INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance. Decide the one claim you are given.

Read the claim in claims/, look up its policy in policies/ and its customer in customers/, and check policy_wording.md and claims_manual.md for the rules that apply.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, for one of the reasons the manual gives.

Write the decision to decisions/<claim id>.json with the fields claim_id, decision, payout (in dollars, 0 unless you approve) and rationale (2-3 sentences to explain the decision along with the relevant section of the wording or the manual).
Then reply with one line: the claim id, the decision and the payout.
"""

## Part 2: The one change

`AgentDefinition` takes a `model`. Its default, `"inherit"`, gives the subagent the main agent's model, which is what the lab did. Passing a model ID puts only the subagents on that model.

Everything else is the lab's: the tools, the instructions, the effort and the one-dollar budget.

In [ ]:
MAIN_MODEL = "claude-sonnet-5-5"
HAIKU = "claude-haiku-5-5"
TOOLS = ["Read", "Glob", "Grep", "Write", "Task", "TaskCreate", "TaskUpdate", "TaskList", "TaskGet"]


def make_options(desk, handler_model):
    """The lab's options, with the claim-handler subagent on handler_model."""
    claim_handler = AgentDefinition(
        description="Decides one claim and writes its decision file. Give it the claim id.",
        prompt=HANDLER_INSTRUCTIONS,
        tools=["Read", "Glob", "Grep", "Write"],
        model=handler_model,
    )
    return ClaudeAgentOptions(
        model=MAIN_MODEL,
        effort="medium",
        cwd=str(desk),
        system_prompt={"type": "preset", "preset": "claude_code", "append": INSTRUCTIONS},
        tools=TOOLS,
        allowed_tools=TOOLS,
        agents={"claim-handler": claim_handler},
        permission_mode="dontAsk",
        setting_sources=[],
        env={"CLAUDE_CODE_DISABLE_AUTO_MEMORY": "1", "ENABLE_CLAUDEAI_MCP_SERVERS": "false"},
        max_budget_usd=1.00,
    )

### One adjustment for Windows

The Claude Agent SDK runs Claude Code as a separate program. Inside a Jupyter notebook on Windows, the notebook's event loop can't start other programs, so `query` fails with `CLIConnectionError: Failed to start Claude Code`. The cell below gives `query` an event loop of its own in a background thread, which lets the cells below use it exactly as the SDK documents it. On Mac and Linux it does nothing at all, so run it either way and carry on.

In [ ]:
import sys

if sys.platform == "win32":
    import asyncio
    import threading

    import claude_agent_sdk

    async def query(*, prompt, options):
        """The SDK's query, run on an event loop of its own in a thread, passing each message back here."""
        loop = asyncio.get_running_loop()
        messages = asyncio.Queue()

        async def run():
            try:
                async for message in claude_agent_sdk.query(prompt=prompt, options=options):
                    loop.call_soon_threadsafe(messages.put_nowait, message)
            except Exception as error:
                loop.call_soon_threadsafe(messages.put_nowait, error)
            loop.call_soon_threadsafe(messages.put_nowait, None)

        threading.Thread(target=asyncio.run, args=(run(),), kwargs={"loop_factory": asyncio.ProactorEventLoop}).start()
        while (message := await messages.get()) is not None:
            if isinstance(message, Exception):
                raise message
            yield message

    print("Applied the Windows adjustment")
else:
    print("Not Windows, so nothing to do here")

## Part 3: Working the queue twice

The same loop as the lab, wrapped in a function so we can run it once for each set of workers. The subagents' steps are indented.

In [ ]:
def describe(tool_call):
    """A short line about one tool call."""
    args = tool_call.input
    if "file_path" in args:
        return f"{tool_call.name} {Path(args['file_path']).name}"
    if tool_call.name in ("Agent", "Task"):
        return f"{tool_call.name}: {args.get('description')}"
    if tool_call.name == "TaskCreate":
        return f"TaskCreate: {args.get('subject')}"
    if tool_call.name == "TaskUpdate":
        return f"TaskUpdate: task {args.get('taskId')} {args.get('status', '')}"
    if "pattern" in args:
        return f"{tool_call.name} {args['pattern']}"
    return tool_call.name

In [ ]:
async def work_the_queue(desk, handler_model):
    """Build a fresh desk, let the agent decide every claim on it, and return its result and the seconds it took."""
    make_desk(desk, claims)
    result = None
    start = time.time()
    async for message in query(prompt="Please work the queue.", options=make_options(desk, handler_model)):
        if isinstance(message, AssistantMessage):
            indent = "      " if message.parent_tool_use_id else ""
            for block in message.content:
                if isinstance(block, ToolUseBlock):
                    print(indent + describe(block))
                elif isinstance(block, TextBlock) and block.text.strip():
                    print(indent + block.text.strip())
        elif isinstance(message, ResultMessage):
            result = message
    seconds = time.time() - start
    print(f"\nFinished in {seconds:.0f} seconds for ${result.total_cost_usd:.2f}")
    return result, seconds

First, every agent on Sonnet, as in the lab.

In [ ]:
SONNET_DESK = Path("sonnet_desk")
sonnet_result, sonnet_seconds = await work_the_queue(SONNET_DESK, "inherit")

Now the same queue with Haiku workers.

In [ ]:
HAIKU_DESK = Path("haiku_desk")
haiku_result, haiku_seconds = await work_the_queue(HAIKU_DESK, HAIKU)

## Part 4: How they compare

The same test as the lab: the decision has to match the handler's and the payout has to be within a dollar.

In [ ]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer.get("decision") == label["decision"] and abs(float(answer.get("payout") or 0) - label["payout"]) <= 1


def read_decision(desk, claim_id):
    """The decision file the agent wrote for one claim, or a missing decision."""
    path = desk / "decisions" / f"{claim_id}.json"
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {"decision": "missing", "payout": 0}


runs = {"Sonnet workers": (SONNET_DESK, sonnet_result, sonnet_seconds), "Haiku workers": (HAIKU_DESK, haiku_result, haiku_seconds)}

rows = []
for name, (desk, result, seconds) in runs.items():
    right = sum(is_right(read_decision(desk, claim_id), label) for claim_id, label in labels.items())
    rows.append({
        "workers": name,
        "right": f"{right} of {len(labels)}",
        "cost": f"${result.total_cost_usd:.2f}",
        "seconds": round(seconds),
    })

pd.DataFrame(rows)

Claim by claim, next to the handler's decision.

In [ ]:
rows = []
for claim_id, label in labels.items():
    row = {"claim": claim_id, "handler": f"{label['decision']} ${label['payout']:,.0f}"}
    for name, (desk, _, _) in runs.items():
        answer = read_decision(desk, claim_id)
        mark = "" if is_right(answer, label) else " (wrong)"
        row[name] = f"{answer.get('decision')} ${float(answer.get('payout') or 0):,.0f}{mark}"
    rows.append(row)

pd.DataFrame(rows)

### Where the money went

`ResultMessage.model_usage` splits the tokens and the cost by model, which also confirms that the workers really ran on Haiku.

In [ ]:
rows = []
for name, (_, result, _) in runs.items():
    for model, usage in (result.model_usage or {}).items():
        rows.append({
            "run": name,
            "model": model,
            "input tokens": usage["inputTokens"] + usage["cacheReadInputTokens"] + usage["cacheCreationInputTokens"],
            "output tokens": usage["outputTokens"],
            "cost": f"${usage['costUSD']:.3f}",
        })

pd.DataFrame(rows)

And the handover note from the Haiku run.

In [ ]:
display(Markdown((HAIKU_DESK / "handover.md").read_text(encoding="utf-8")))

## What I found

| | Sonnet workers | Haiku workers |
|---|---|---|
| Right | 5 of 5 | 5 of 5 |
| Cost | $0.36 | $0.10 |
| Time | 36 seconds | 39 seconds |

- **The cost fell by about 70%, with no loss of accuracy on these five claims.** The Haiku workers decided all five as the handlers did, including CLM-1017, the one that needs the claims manual.
- **The workers were where the money went.** With Sonnet workers, Sonnet read about 516,000 input tokens. With Haiku workers, Haiku did the reading, about 288,000 input tokens, for under 2 cents.
- **Now the manager is the expensive part.** In the Haiku run, Sonnet as the main agent cost $0.085 of the $0.10. It also did more this time: it read all five decision files before writing the handover, which it didn't do in the Sonnet run. Next to try: Haiku for the manager too.
- **The time barely changed**, because the five workers run in parallel and the slowest one sets the pace.
- Even the all-Sonnet run shows a little Haiku: Claude Code made one small Haiku call of its own.

**A caveat:** five claims and one run each is a small test, and a model can get lucky on five. Before trusting Haiku workers with the real queue, I'd run the 25-claim queue (`tiny=False`, with `max_budget_usd` raised) or the held-out claims a few times and compare again.